In [ ]:
import yfinance as yf
import numpy as np
import statsmodels.api as sm
import statsmodels.tsa.stattools as ts
import itertools
from scipy.stats import norm
from scipy.optimize import minimize


In [ ]:
transaction_tickers = ['CPAY', 'FIS', 'FISV', 'GPN', 'JKHY', 'MA', 'PYPL', 'V', 'XYZ']
def download_data(stock, start, end):
    try:
        df = yf.download(stock, start=start, end=end)
        return df['Close']
    except Exception as e:
        print(f"Error downloading {stock}: {e}")
        return None

data = download_data(transaction_tickers, start='2016-01-01', end='2022-12-31')
data = data.dropna(axis=1)



In [ ]:
DT = 1/252
def fit_ou_halflife(spread_values):
    sx = spread_values[:-1]
    sy = spread_values[1:]

    def neg_ll(params):
        kappa, mu, sigma = params
        if kappa <= 0 or sigma <= 0:
            return 1e12
        e   = np.exp(-kappa * DT)
        c_m = sx * e + mu * (1 - e)
        c_v = (sigma**2 / (2 * kappa)) * (1 - e**2)
        if c_v <= 0:
            return 1e12
        return -norm.logpdf(sy, c_m, np.sqrt(c_v)).sum()

    coef = np.polyfit(sx, sy - sx, 1)[0]
    k0   = max(-coef / DT, 0.5) if coef < 0 else 1.5
    m0   = float(spread_values.mean())
    s0   = float(spread_values.std()) * np.sqrt(2 * k0)

    res = minimize(neg_ll, [k0, m0, s0],
                   bounds=[(0.01, 50), (None, None), (1e-6, None)],
                   method='L-BFGS-B')

    kappa = res.x[0]
    half_life = np.log(2) / kappa * 252
    return half_life


total_pairs = 0
passed_corr = 0
passed_coint = 0
passed_hurst = 0
passed_halflife = 0

viable_pairs = []

for stock_a, stock_b in itertools.combinations(data.columns, 2):
    total_pairs += 1
    
    df = np.log(data[[stock_a, stock_b]].dropna())
    
    # Correlation Filter
    corr = df[stock_a].corr(df[stock_b])
    if corr < 0.70: continue
    passed_corr += 1
        
    # Cointegration 95% confidence filter
    score, p_value, _ = ts.coint(df[stock_a], df[stock_b])
    if p_value >= 0.05: continue
    passed_coint += 1
        
    # Calculate Spread
    X = sm.add_constant(df[stock_b])
    model = sm.OLS(df[stock_a], X).fit()
    alpha = model.params.iloc[0]
    beta = model.params.iloc[1]
    spread = df[stock_a] - (beta * df[stock_b])

    # Hurst Exponent Filter
    # hurst = calculate_hurst_exponent(spread.values)
    # if hurst >= 0.50: continue
    # passed_hurst += 1
        
    # Half-Life Filter
    half_life = fit_ou_halflife(spread.values)
    if not (1 <= half_life <= 90): continue
    passed_halflife += 1
        
    viable_pairs.append(f"{stock_a} / {stock_b} (p-val: {p_value:.3f}, HL: {half_life:.1f}), Corr: {corr:.2f}")

print(f"Total Combinations Tested: {total_pairs}")
print(f"Passed Correlation (>0.70):  {passed_corr}")
print(f"Passed Cointegration (<0.05):{passed_coint}")
# print(f"Passed Hurst (<0.50):        {passed_hurst}")
print(f"Passed Half-Life (1-90d):    {passed_halflife}")

print(f"\nTotal Surviving Pairs: {len(viable_pairs)}")
for p in viable_pairs:
    print(p)